In [9]:
import pandas as pd
import re

N_GENS = 11
N_CHAINS = 10

def clean_text(x):
    if pd.isna(x):
        return pd.NA
    return re.sub(r"\s+", " ", str(x)).strip()

def clean(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].map(clean_text)
    return df

def get_idx(row, prev, prev_gen):
    if pd.notna(row["Original word"]):
        match = prev.loc[
            prev[f"Word_Gen_{prev_gen}"] == row["Original word"],
            "idx"
        ]
    else:
        match = prev.loc[
            prev[f"Description_Gen_{prev_gen}"] == row["Original description"],
            "idx"
        ]

    return match.iloc[0]

for chain in range(1, N_CHAINS + 1):

    prev = pd.read_csv(f"Gen1/Gen_1_Chain_{chain}.csv")
    prev = clean(prev)
    prev["idx"] = prev.index
    prev.to_csv(f"Gen1/Gen_1_Chain_{chain}.csv", index=False)

    for gen in range(2, N_GENS + 1):
        path = f"Gen{gen}/Gen_{gen}_Chain_{chain}.csv"

        df = pd.read_csv(path)
        df = clean(df)

        if f"Word_Gen_{gen}" not in df.columns:
            df = df.rename(columns={"Guessed word": f"Word_Gen_{gen}"})

        if f"Description_Gen_{gen}" not in df.columns:
            df = df.rename(columns={"Guessed description": f"Description_Gen_{gen}"})

        df["idx"] = df.apply(lambda row: get_idx(row, prev, gen - 1), axis=1)

        df.to_csv(path, index=False)
        prev = df

    print(f"Finished Chain {chain}")

Finished Chain 1
Finished Chain 2
Finished Chain 3
Finished Chain 4
Finished Chain 5
Finished Chain 6
Finished Chain 7
Finished Chain 8
Finished Chain 9
Finished Chain 10


In [31]:
import pandas as pd
from pathlib import Path

N_GENS = 11
N_CHAINS = 10

Path("merged").mkdir(exist_ok=True)

for chain in range(1, N_CHAINS + 1):

    merged = pd.read_csv(f"Gen1/Gen_1_Chain_{chain}.csv")[
        ["idx", "Category", "Word_Gen_1", "Description_Gen_1"]
    ]

    for gen in range(2, N_GENS + 1):
        df = pd.read_csv(f"Gen{gen}/Gen_{gen}_Chain_{chain}.csv")[
            ["idx", f"Word_Gen_{gen}", f"Description_Gen_{gen}"]
        ]

        merged = merged.merge(df, on="idx", how="outer")

    merged = merged.sort_values("idx")

    merged.to_csv(f"merged/Chain_{chain}_merged.csv", index=False)

    print(f"Saved merged/Chain_{chain}_merged.csv")

Saved merged/Chain_1_merged.csv
Saved merged/Chain_2_merged.csv
Saved merged/Chain_3_merged.csv
Saved merged/Chain_4_merged.csv
Saved merged/Chain_5_merged.csv
Saved merged/Chain_6_merged.csv
Saved merged/Chain_7_merged.csv
Saved merged/Chain_8_merged.csv
Saved merged/Chain_9_merged.csv
Saved merged/Chain_10_merged.csv


In [30]:
import pandas as pd
import re
from pathlib import Path

input_dir = Path("merged")
output_dir = Path("merged_with_counts")
output_dir.mkdir(exist_ok=True)

def is_correct_guess(target, guess):
    if pd.isna(target) or pd.isna(guess):
        return False

    target = str(target).strip()
    guess = str(guess).strip()

    # Exact target, optional plural "s", then optional punctuation
    pattern = rf"^{re.escape(target)}s?\W*$"

    return re.fullmatch(pattern, guess, flags=re.IGNORECASE) is not None

for chain_id in range(1, 11):
    input_path = input_dir / f"Chain_{chain_id}_merged.csv"
    output_path = output_dir / f"Chain_{chain_id}_merged_with_counts.csv"

    df = pd.read_csv(input_path)

    df.drop(columns=[f"Description_Gen_{i}" for i in range(1, 12)], inplace=True)

    future_word_cols = [f"Word_Gen_{i}" for i in range(2, 12)]

    for i, col in enumerate(future_word_cols, start=2):
        previous_cols = [f"Word_Gen_{j}" for j in range(2, i + 1)]

        df[f"Correct_Until_Gen_{i}"] = df.apply(
            lambda row: sum(
                is_correct_guess(row["Word_Gen_1"], row[future_col])
                for future_col in previous_cols
            ),
            axis=1
        )

    df.rename(
        columns={f"Word_Gen_{i}": f"Gen_{i}" for i in range(1, 12)},
        inplace=True
    )
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: merged_with_counts/Chain_1_merged_with_counts.csv
Saved: merged_with_counts/Chain_2_merged_with_counts.csv
Saved: merged_with_counts/Chain_3_merged_with_counts.csv
Saved: merged_with_counts/Chain_4_merged_with_counts.csv
Saved: merged_with_counts/Chain_5_merged_with_counts.csv
Saved: merged_with_counts/Chain_6_merged_with_counts.csv
Saved: merged_with_counts/Chain_7_merged_with_counts.csv
Saved: merged_with_counts/Chain_8_merged_with_counts.csv
Saved: merged_with_counts/Chain_9_merged_with_counts.csv
Saved: merged_with_counts/Chain_10_merged_with_counts.csv


In [29]:
import pandas as pd
from pathlib import Path

input_dir = Path("merged_with_counts")
output_dir = Path("category_sums")
output_dir.mkdir(exist_ok=True)

for chain_id in range(1, 11):
    input_path = input_dir / f"Chain_{chain_id}_merged_with_counts.csv"
    output_path = output_dir / f"Chain_{chain_id}_category_sums.csv"

    df = pd.read_csv(input_path)

    count_cols = [col for col in df.columns if col.startswith("Correct_Until_Gen_")]

    category_sums = (
        df.groupby("Category")[count_cols]
          .sum()
          .reset_index()
    )

    category_sums.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: category_sums/Chain_1_category_sums.csv
Saved: category_sums/Chain_2_category_sums.csv
Saved: category_sums/Chain_3_category_sums.csv
Saved: category_sums/Chain_4_category_sums.csv
Saved: category_sums/Chain_5_category_sums.csv
Saved: category_sums/Chain_6_category_sums.csv
Saved: category_sums/Chain_7_category_sums.csv
Saved: category_sums/Chain_8_category_sums.csv
Saved: category_sums/Chain_9_category_sums.csv
Saved: category_sums/Chain_10_category_sums.csv


In [23]:
category_sums

,Category,Correct_Until_Gen_2,Correct_Until_Gen_3,Correct_Until_Gen_4,Correct_Until_Gen_5,Correct_Until_Gen_6,Correct_Until_Gen_7,Correct_Until_Gen_8,Correct_Until_Gen_9,Correct_Until_Gen_10,Correct_Until_Gen_11
0,High-Freq-Abstract,2,5,7,10,12,14,14,15,15,15
1,High-Freq-Concrete,3,7,10,14,17,21,23,27,29,32
2,Low-Freq-Abstract,1,2,3,4,4,4,4,4,4,4
3,Low-Freq-Concrete,3,5,8,10,13,15,15,17,17,19


In [26]:
import pandas as pd
from pathlib import Path

input_dir = Path("merged_with_counts")
output_path = Path("all_chains_grouped_by_category.csv")

all_dfs = []

for chain_id in range(1, 11):
    input_path = input_dir / f"Chain_{chain_id}_merged_with_counts.csv"

    df = pd.read_csv(input_path)

    all_dfs.append(df)

# Combine all chains into one dataframe
combined_df = pd.concat(all_dfs, ignore_index=True)

# Select the count columns
count_cols = [
    col for col in combined_df.columns
    if col.startswith("Correct_Until_Gen_")
]

# Group by category across all chains
category_sums = (
    combined_df.groupby("Category")[count_cols]
    .sum()
    .reset_index()
)

category_sums.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: all_chains_grouped_by_category.csv


In [28]:
category_sums

,Category,Correct_Until_Gen_2,Correct_Until_Gen_3,Correct_Until_Gen_4,Correct_Until_Gen_5,Correct_Until_Gen_6,Correct_Until_Gen_7,Correct_Until_Gen_8,Correct_Until_Gen_9,Correct_Until_Gen_10,Correct_Until_Gen_11
0,High-Freq-Abstract,29,60,80,107,118,135,145,159,165,177
1,High-Freq-Concrete,37,74,107,143,172,203,226,250,272,295
2,Low-Freq-Abstract,148,156,163,164,166,167,168,169,170,171
3,Low-Freq-Concrete,35,58,78,98,110,126,133,148,153,168
